# BestModel: Ensemble of RoBERTa-CE + RoBERTa-Focal

**Task:** SemEval 2022 Task 4 Subtask 1 — Patronizing and Condescending Language (PCL) Detection  
**Best model:** Probability-averaging ensemble of two RoBERTa-base classifiers fine-tuned with different loss functions.

## Approach Summary

The dataset is heavily class-imbalanced (~9.5% positive). A single loss function creates a precision-recall trade-off:
- **RoBERTa-CE** (cross-entropy): treats all examples equally → conservative, higher precision
- **RoBERTa-Focal** (focal loss, α=0.75, γ=2.0): down-weights easy negatives → higher recall

Averaging their output probabilities balances the two biases and outperforms either model alone:

| Model | Dev F1 | Precision | Recall |
|---|---|---|---|
| RoBERTa-CE | 0.5954 | 0.603 | 0.588 |
| RoBERTa-Focal | 0.5948 | 0.557 | 0.638 |
| **Ensemble** | **0.5967** | 0.568 | 0.628 |

DeBERTa-v3-base was also tested but collapsed to predicting all samples as positive (F1=0.173) due to instability under severe class imbalance.

Full implementation: [`stage4/stage4_final.py`](../stage4/stage4_final.py)

---

## Prerequisites
- Python 3.10+, CUDA-capable GPU
- Dataset files placed in:
  - `dataset/train/data/dontpatronizeme_pcl.tsv`
  - `dataset/train/labels/train_semeval_parids-labels.csv`
  - `dataset/train/labels/dev_semeval_parids-labels.csv`
  - `dataset/test/data/task4_test.tsv`

## 1. Setup

In [ ]:
# Clone the repository (skip if already cloned)
!git clone https://gitlab.doc.ic.ac.uk/ct1222/nlp.git
%cd nlp

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Hyperparameter Optimisation (HPO)

Runs Bayesian HPO (5 trials each) on an internal 85/15 stratified split of the training set.  
Saves the best checkpoint for each approach to `stage4/outputs_stage4/`.

In [ ]:
!python stage4/stage4_final.py --mode hpo --n_trials 5

## 3. Compare Models on Official Dev Set

Evaluates HPO checkpoints on the official dev set and selects the best approach.  
Results saved to `stage4/outputs_stage4/comparison_results.json`.

In [ ]:
!python stage4/stage4_final.py --mode compare

## 4. Retrain on Full Data

- **dev.txt**: reuses HPO checkpoints (trained on 85% train, no dev leakage). Threshold tuned on internal 15% split.
- **test.txt**: retrains on train+dev combined. Uses a 95/5 holdout to find the best epoch, then retrains on 100% for that many epochs. Threshold tuned on official dev.

In [ ]:
!python stage4/stage4_final.py --mode retrain

## 5. Generate Predictions

Outputs `dev.txt` (2094 lines) and `test.txt` (3832 lines) to `stage4/outputs_stage4/`.

In [ ]:
!python stage4/stage4_final.py --mode predict

In [ ]:
# Verify output line counts
!wc -l stage4/outputs_stage4/dev.txt stage4/outputs_stage4/test.txt
# Predictions are also committed to predictions/ at the repo root for easy access

## 6. Results

View the comparison results across all models:

In [ ]:
import json

with open('stage4/outputs_stage4/comparison_results.json') as f:
    results = json.load(f)

print(f"{'Model':<20} {'Dev F1':>8} {'Precision':>10} {'Recall':>8}")
print('-' * 50)
for name, r in results.items():
    if name.startswith('_'):
        continue
    m = r['metrics']
    marker = ' ★' if name == results.get('_best_approach') else ''
    print(f"{name + marker:<20} {r['official_dev_f1']:>8.4f} {m['precision']:>10.4f} {m['recall']:>8.4f}")